### Feature Engineering

In [72]:
import pandas as pd

df = pd.read_csv('../data/02_interim/03_clean_races_info.csv', parse_dates=['date', 'dob'])

#### 1. Adding Driver Age at Race Date to Dataset

In [73]:
target_idx = df.columns.get_loc('driverId') + 1

df.insert(target_idx, 'driver_age', (df['date'] - df['dob']).dt.days / 365.25)

#### 2. Adding Driver Momentum (Last 3 Races)

In [74]:
df.insert(target_idx + 1, 'driver_momentum',
          df.groupby('driverId')['positionOrder'].transform(
              lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
          ))

In [75]:
df['driver_momentum'] = df['driver_momentum'].fillna(df['grid'])

df[['driverId', 'raceId', 'positionOrder', 'grid', 'driver_momentum']]

,driverId,raceId,positionOrder,grid,driver_momentum
0,30,53,2,1,1.000000
1,13,53,9,2,2.000000
2,18,53,4,3,3.000000
3,4,53,1,4,4.000000
4,31,53,5,5,5.000000
...,...,...,...,...,...
7885,825,1144,16,14,9.333333
7886,848,1144,11,18,17.333333
7887,855,1144,13,15,12.000000
7888,862,1144,15,17,17.000000


#### 3. Adding Constructor/Team Momentum

In [76]:
# Constructor by race mean
team_avg = df.groupby(['constructorId', 'date'])['positionOrder'].mean().reset_index()

team_avg['constructor_momentum'] = team_avg.groupby('constructorId')['positionOrder'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

df = pd.merge(df, team_avg[['constructorId', 'date', 'constructor_momentum']], on=['constructorId', 'date'], how='left', validate='one_to_one')

In [77]:
df['constructor_momentum'] = df['constructor_momentum'].fillna(
    df.groupby(['constructorId', 'raceId'])['grid'].transform('mean')
)

,statusId,status,status_group,qualifyId,raceId,driverId,driver_age,driver_momentum,position_qualifying,q1,...,dob,year,round,circuitId,date,resultId,grid,positionOrder,laps,constructor_momentum
0,1,Finished,Finished,743,53,30,37.185489,1.000000,1,1:33.310,...,1969-01-03,2006,1,3,2006-03-12,744,1,2,57,1.500000
1,1,Finished,Finished,744,53,13,24.878850,2.000000,2,1:33.579,...,1981-04-25,2006,1,3,2006-03-12,751,2,9,57,1.500000
2,1,Finished,Finished,745,53,18,26.143737,3.000000,3,1:32.603,...,1980-01-19,2006,1,3,2006-03-12,746,3,4,57,4.500000
3,1,Finished,Finished,746,53,4,24.618754,4.000000,4,1:32.433,...,1981-07-29,2006,1,3,2006-03-12,743,4,1,57,6.500000
4,1,Finished,Finished,747,53,31,30.475017,5.000000,5,1:33.233,...,1975-09-20,2006,1,3,2006-03-12,747,5,5,57,13.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7885,11,+1 Lap,Finished,10546,1144,825,32.175222,9.333333,15,1:23.632,...,1992-10-05,2024,24,24,2024-12-08,26760,14,16,57,12.833333
7886,11,+1 Lap,Finished,10547,1144,848,28.711841,17.333333,16,1:23.821,...,1996-03-23,2024,24,24,2024-12-08,26755,18,11,57,17.000000
7887,11,+1 Lap,Finished,10548,1144,855,25.527721,12.000000,17,1:23.880,...,1999-05-30,2024,24,24,2024-12-08,26757,15,13,57,13.000000
7888,11,+1 Lap,Finished,10551,1144,862,21.883641,17.000000,20,1:24.105,...,2003-01-20,2024,24,24,2024-12-08,26759,17,15,57,11.166667


In [78]:
cols = df.columns.tolist()
col_to_move = cols.pop(cols.index('constructor_momentum'))

target_idx = cols.index('constructorId') + 1

cols.insert(target_idx, col_to_move)

df = df[cols]

In [79]:
df

,statusId,status,status_group,qualifyId,raceId,driverId,driver_age,driver_momentum,position_qualifying,q1,...,constructor_momentum,dob,year,round,circuitId,date,resultId,grid,positionOrder,laps
0,1,Finished,Finished,743,53,30,37.185489,1.000000,1,1:33.310,...,1.500000,1969-01-03,2006,1,3,2006-03-12,744,1,2,57
1,1,Finished,Finished,744,53,13,24.878850,2.000000,2,1:33.579,...,1.500000,1981-04-25,2006,1,3,2006-03-12,751,2,9,57
2,1,Finished,Finished,745,53,18,26.143737,3.000000,3,1:32.603,...,4.500000,1980-01-19,2006,1,3,2006-03-12,746,3,4,57
3,1,Finished,Finished,746,53,4,24.618754,4.000000,4,1:32.433,...,6.500000,1981-07-29,2006,1,3,2006-03-12,743,4,1,57
4,1,Finished,Finished,747,53,31,30.475017,5.000000,5,1:33.233,...,13.500000,1975-09-20,2006,1,3,2006-03-12,747,5,5,57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7885,11,+1 Lap,Finished,10546,1144,825,32.175222,9.333333,15,1:23.632,...,12.833333,1992-10-05,2024,24,24,2024-12-08,26760,14,16,57
7886,11,+1 Lap,Finished,10547,1144,848,28.711841,17.333333,16,1:23.821,...,17.000000,1996-03-23,2024,24,24,2024-12-08,26755,18,11,57
7887,11,+1 Lap,Finished,10548,1144,855,25.527721,12.000000,17,1:23.880,...,13.000000,1999-05-30,2024,24,24,2024-12-08,26757,15,13,57
7888,11,+1 Lap,Finished,10551,1144,862,21.883641,17.000000,20,1:24.105,...,11.166667,2003-01-20,2024,24,24,2024-12-08,26759,17,15,57
